# Build `paired_edit_phase1_expand_batch1`

Terminal refresh command:
```bash
rm -rf /content/nail-retouch-assistant
git clone https://github.com/Zifpen/nail-retouch-assistant.git /content/nail-retouch-assistant
cd /content/nail-retouch-assistant
git checkout main
```


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import yaml

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/nail-retouch-assistant')
CONFIG_PATH = PROJECT_ROOT / 'colab' / 'paired_edit_phase1_expand_batch1_oldroute_config.yaml'
MANIFEST_PATH = PROJECT_ROOT / 'dataset' / 'annotations' / 'paired_edit_phase1_expand_batch1_manifest.json'
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))

DRIVE_RAW_DIR = Path(cfg.get('drive_raw_dir', '/content/drive/MyDrive/nail-retouch-raw'))
DRIVE_DATASET_DIR = Path(cfg['drive_dataset_dir'])
LOCAL_DATASET_DIR = PROJECT_ROOT / cfg['dataset_dir']

if not DRIVE_RAW_DIR.exists():
    raise FileNotFoundError(f'Missing raw pair directory in Drive: {DRIVE_RAW_DIR}')
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Missing manifest: {MANIFEST_PATH}')

print('raw source:', DRIVE_RAW_DIR)
print('manifest:', MANIFEST_PATH)
print('drive dataset output:', DRIVE_DATASET_DIR)
print('local dataset output:', LOCAL_DATASET_DIR)


In [ ]:
DRIVE_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.rmtree(DRIVE_DATASET_DIR, ignore_errors=True)
shutil.rmtree(LOCAL_DATASET_DIR, ignore_errors=True)

cmd = [
    'python3',
    str(PROJECT_ROOT / 'src' / 'data' / 'build_curated_paired_edit_dataset.py'),
    '--raw-dir', str(DRIVE_RAW_DIR),
    '--manifest', str(MANIFEST_PATH),
    '--output-dir', str(DRIVE_DATASET_DIR),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

shutil.copytree(DRIVE_DATASET_DIR, LOCAL_DATASET_DIR)
print('Built dataset in Drive and copied locally.')


In [ ]:
for rel in ('train_A', 'train_B', 'test_A', 'test_B', 'train_prompts.json', 'test_prompts.json'):
    path = DRIVE_DATASET_DIR / rel
    print(rel, 'exists' if path.exists() else 'missing')
    if path.is_dir():
        print(' count:', len(list(path.glob('*'))))
